# PROYECTO: ACA Nº 1 - APRENDIZAJE AUTOMÁTICO
## Modelado con Algoritmos Clásicos de Machine Learning y Despliegue en Streamlit

**Corporación Unificada Nacional de Educación Superior (CUN)**  
**Escuela de Ingeniería | Especialización en Inteligencia Artificial**  
**Asignatura:** Aprendizaje Automático  
**Docente:** Dr. Hamilton Rivera Flor  

---
### Integrantes:
* **Anyela María Torres Sánchez**
* **Brandon Andres Ardila Meta**
* **Edwin Arturo Contreras Padua**

**Pilar Temático:** Ventas (Sector Inmobiliario / Comercial)  
**Dataset Seleccionado:** California Housing (`housing.csv`)


## 1. Introducción y Situación Problema

### Situación Problema / Pregunta Orientadora
> *¿Cómo transformar datos del sector de ventas inmobiliarias en conocimiento predictivo mediante algoritmos clásicos de Machine Learning, permitiendo estimar el precio comercial de una propiedad de manera confiable e interactiva para usuarios finales?*

### Objetivo General
Desarrollar una solución integral de Machine Learning clásico aplicada a la predicción de precios en el sector inmobiliario comercial, abarcando desde la exploración y limpieza de datos hasta el entrenamiento de modelos tradicionales de regresión y su posterior despliegue en una aplicación web interactiva con Streamlit.


## 2. Carga de Librerías y Datos
En esta etapa cargamos las herramientas básicas de ciencia de datos:
* `pandas` y `numpy` para manipulación de tablas y operaciones numéricas.
* `matplotlib` y `seaborn` para gráficos visuales sencillos.
* `scikit-learn` para el preprocesamiento, modelado clásico y cálculo de métricas.

> **Nota de compatibilidad:** El código verifica automáticamente si se está ejecutando de forma local (en la carpeta del proyecto) o en la nube (**Google Colab**). En Colab descarga el dataset oficial directamente sin requerir subir archivos manualmente.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Configuración gráfica básica
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

# Carga inteligente: Local o Google Colab
ruta_local = os.path.join("..", "data", "housing.csv")
url_oficial = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

if os.path.exists(ruta_local):
    print("Cargando dataset desde ruta local del proyecto...")
    df = pd.read_csv(ruta_local)
elif os.path.exists("housing.csv"):
    print("Cargando dataset desde directorio actual...")
    df = pd.read_csv("housing.csv")
else:
    print("Ejecutando en Google Colab: Descargando dataset oficial desde repositorio público...")
    df = pd.read_csv(url_oficial)

print(f"Dataset cargado con éxito: {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()


## 3. Paso 1: Exploración y Análisis Exploratorio de Datos (EDA)
Revisamos los tipos de variables, las estadísticas básicas y la presencia de valores faltantes.


In [ ]:
# Información general de columnas y tipos de datos
df.info()


In [ ]:
# Verificación explícita de valores faltantes (nulos)
nulos = df.isnull().sum()
print("Valores nulos por columna:")
print(nulos[nulos > 0])


In [ ]:
# Resumen estadístico de las variables numéricas
df.describe().round(2)


### Visualización 1: Distribución del Precio de las Casas (`median_house_value`)
Queremos conocer cómo se distribuye el valor comercial de las propiedades. Observamos el rango de precios y si existen acumulaciones particulares.


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["median_house_value"], bins=40, kde=True, color="steelblue")
plt.title("Distribución del Precio Medio de la Vivienda (USD)")
plt.xlabel("Precio ($ USD)")
plt.ylabel("Frecuencia")
plt.show()


### Visualización 2: Relación entre Ingreso Medio y Precio de la Vivienda
Es lógico pensar que en zonas donde las familias tienen mayores ingresos, las casas tienen mayor valor comercial. Verificamos esta relación mediante un gráfico de dispersión.


In [ ]:
plt.figure(figsize=(8, 4.5))
# Tomamos una muestra aleatoria de 2,000 puntos para una visualización clara
muestra = df.sample(2000, random_state=42)
sns.scatterplot(data=muestra, x="median_income", y="median_house_value", alpha=0.5, color="darkorange")
plt.title("Ingreso Medio del Distrito vs. Precio Medio de la Vivienda")
plt.xlabel("Ingreso Medio (en decenas de miles de USD)")
plt.ylabel("Precio ($ USD)")
plt.show()


### Visualización 3: Ubicación Geográfica y Categoría de Cercanía al Mar
La variable categórica `ocean_proximity` clasifica la cercanía de la propiedad a la costa. Graficamos la latitud y longitud coloreadas por dicha categoría.


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=muestra, x="longitude", y="latitude", hue="ocean_proximity", palette="Set2", alpha=0.7)
plt.title("Ubicación Geográfica según Proximidad al Océano")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 4. Paso 2: Limpieza y Preparación de Datos

En esta etapa abordamos los dos aspectos clave identificados en el EDA:
1. **Imputación de valores nulos:** La variable `total_bedrooms` tiene 207 datos faltantes. Los imputamos utilizando la **mediana**, una técnica robusta ante posibles valores extremos.
2. **Codificación de variables categóricas:** La columna `ocean_proximity` es de tipo texto. Aplicamos **One-Hot Encoding** para convertirla en variables numéricas binarias.
3. **Escalamiento:** Estandarizamos las variables numéricas (`StandardScaler`) para que queden en una escala comparable con media 0 y varianza 1.
4. **Partición de datos:** Dividimos el dataset en 80% para entrenamiento (`train`) y 20% para evaluación final (`test`).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separamos las características predictoras (X) de la variable objetivo (y)
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

# Identificamos columnas numéricas y categóricas
columnas_num = [
    "longitude", "latitude", "housing_median_age",
    "total_rooms", "total_bedrooms", "population",
    "households", "median_income"
]
columnas_cat = ["ocean_proximity"]

# Sub-pipeline para números: Imputación con mediana + Escalamiento estándar
pipeline_numerico = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler())
])

# Transformador general unificado
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, columnas_num),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), columnas_cat)
])

# Partición en 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Filas de entrenamiento: {X_train.shape[0]}")
print(f"Filas de prueba: {X_test.shape[0]}")


## 5. Paso 3: Modelado Clásico de Machine Learning

Para contrastar diferentes enfoques clásicos de aprendizaje supervisado, seleccionamos dos algoritmos tradicionales:
1. **Regresión Lineal:** Modelo paramétrico base que busca una relación lineal directa entre los atributos y el precio.
2. **Random Forest Regressor (Bosque Aleatorio):** Modelo de ensamble basado en árboles de decisión que combina múltiples predicciones para capturar patrones no lineales sin complejidad innecesaria.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Modelo 1: Regresión Lineal
modelo_lineal = Pipeline([
    ("prep", preprocesador),
    ("regresor", LinearRegression())
])

modelo_lineal.fit(X_train, y_train)
print("Modelo 1 (Regresión Lineal) entrenado exitosamente.")


In [ ]:
# Modelo 2: Random Forest Regressor
modelo_rf = Pipeline([
    ("prep", preprocesador),
    ("regresor", RandomForestRegressor(n_estimators=80, random_state=42, n_jobs=-1))
])

modelo_rf.fit(X_train, y_train)
print("Modelo 2 (Random Forest) entrenado exitosamente.")


## 6. Paso 4: Evaluación Cuantitativa de Métricas

Evaluamos el desempeño de ambos modelos sobre el conjunto de prueba (`X_test`), utilizando métricas estándar de regresión:
* **$R^2$ (Coeficiente de Determinación):** Indica qué porcentaje de la variación del precio logra explicar el modelo (valores cercanos a 1.0 son deseables).
* **MAE (Error Absoluto Medio):** Es el promedio en dólares del desvío de la predicción respecto al valor real.
* **RMSE (Raíz del Error Cuadrático Medio):** Mide la magnitud de los errores, penalizando con mayor fuerza las desviaciones grandes.


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Predicciones con ambos modelos
y_pred_lin = modelo_lineal.predict(X_test)
y_pred_rf = modelo_rf.predict(X_test)

# Métricas Modelo Lineal
r2_lin = r2_score(y_test, y_pred_lin)
mae_lin = mean_absolute_error(y_test, y_pred_lin)
rmse_lin = root_mean_squared_error(y_test, y_pred_lin)

# Métricas Random Forest
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)

# Tabla comparativa de resultados
tabla_resultados = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Random Forest Regressor"],
    "R² Score": [round(r2_lin, 4), round(r2_rf, 4)],
    "MAE ($ USD)": [round(mae_lin, 2), round(mae_rf, 2)],
    "RMSE ($ USD)": [round(rmse_lin, 2), round(rmse_rf, 2)]
})

print("=== COMPARACIÓN DE DESEMPEÑO ===")
tabla_resultados


### Visualización: Valores Reales vs. Valores Predichos (Random Forest)
Graficamos las predicciones del modelo ganador frente a los valores reales. Entre más cercanos estén los puntos a la línea diagonal roja, más exacta es la estimación.


In [ ]:
plt.figure(figsize=(7, 5))
indices = np.random.RandomState(42).choice(len(y_test), size=250, replace=False)
plt.scatter(y_test.iloc[indices], y_pred_rf[indices], alpha=0.6, color="teal", edgecolors="k", s=40)
plt.plot([0, 500000], [0, 500000], color="red", linestyle="--", linewidth=2, label="Ajuste Ideal")
plt.title("Random Forest: Precios Reales vs. Precios Predichos")
plt.xlabel("Precio Real ($ USD)")
plt.ylabel("Precio Predicho ($ USD)")
plt.legend()
plt.show()


## 7. Conclusiones y Exportación del Modelo

### Conclusiones Técnicas:
1. **Selección del Algoritmo:** El modelo **Random Forest Regressor** demostró un desempeño notablemente superior al de la Regresión Lineal, alcanzando un $R^2$ de **~81.7%** frente al **62.5%** del modelo base, y disminuyendo el error promedio (MAE) de **$50,670 USD** a **$31,672 USD** aproximadamente.
2. **Interpretación:** La relación entre ubicación geográfica, cercanía al mar e ingresos con el precio de las viviendas es no lineal, razón por la cual los árboles de decisión en ensamble se ajustan mucho mejor al comportamiento real del mercado.
3. **Exportación:** Procedemos a serializar y guardar el pipeline completo de Random Forest (`modelo_casas_rf.joblib`) para integrarlo de forma directa en la aplicación de **Streamlit** (Entregable 3).


In [ ]:
# Exportamos el pipeline completo (preprocesamiento + modelo con compresión para GitHub/Cloud)
os.makedirs(os.path.join("..", "models"), exist_ok=True)
ruta_modelo = os.path.join("..", "models", "modelo_casas_rf.joblib")
joblib.dump(modelo_rf, ruta_modelo, compress=3)
print(f"¡Modelo serializado y guardado con éxito en: {ruta_modelo}!")
